In [30]:
import os

In [31]:
#Print working directory

%pwd

'C:\\Practice_Projects\\ML\\App_Dev\\ML_Project_02\\chicken_disease_classification_project'

In [32]:
#Change directory
os.chdir("../")

In [33]:
%pwd

'C:\\Practice_Projects\\ML\\App_Dev\\ML_Project_02'

In [34]:
import os

os.chdir(r"C:\Practice_Projects\ML\App_Dev\ML_Project_02\chicken_disease_classification_project")
print(os.getcwd())

C:\Practice_Projects\ML\App_Dev\ML_Project_02\chicken_disease_classification_project


#### Update the entity

In [35]:
# Entity is the return type of a function. If you don't have in-built return type, it needs to create custom return type.

#So, create data ingestion related config or entity
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:

    #these are getting from config.yaml
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [36]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories

In [37]:


#from pathlib import Path

#CONFIG_FILE_PATH = Path("config/config.yaml")
#PARAMS_FILE_PATH = Path("params.yaml")

In [38]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL, 
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        #Created custom return type
        return data_ingestion_config

#### Update the Components

In [39]:
import os
#for requesting data
import urllib.request as request
#unzip the data
import zipfile
from src.cnnClassifier import logger
from src.cnnClassifier.utils.common import get_size


In [40]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    #Downloading Data
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )

            logger.info(f"{filename} download with following info: \n{headers}")

        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    #Unzip the data
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """

        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)

        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)


#### Create Pipeline

In [41]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise e


[2026-03-02 04:44:06,144: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-02 04:44:06,154: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-02 04:44:06,160: INFO: common: Created directory at: artifacts]
[2026-03-02 04:44:06,162: INFO: common: Created directory at: artifacts/data_ingestion]
[2026-03-02 04:44:10,505: INFO: 993601903: artifacts/data_ingestion/data.zip download with following info: 
Connection: close
Content-Length: 11616915
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "adf745abc03891fe493c3be264ec012691fe3fa21d861f35a27edbe6d86a76b1"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: CFC0:A43FE:CC4863:E54231:69A4C83E
Accept-Ranges: bytes
Date: Sun, 01 Mar 2026 23:14:11 GMT
Via: 1.1 varnish
X-Served-By: cache-sin-wsat1880068-SIN
X-Cac

In [ ]:
# Convert this file to moduler coding

#So, follow the workfloew first